# Geometric EEG SSL — Data Download Notebook (CPU only)

**Purpose:** Cache all three datasets to Google Drive once. No GPU needed —
**do not waste GPU credit running this**. Use a standard CPU runtime.

After this notebook completes:
- PhysioNet MI is enough to start **`colab_pretrain.ipynb`** (all 5 variants).
- BCIC-2B and Sleep-EDFx are only required to run **`colab_experiment.ipynb`**.

Sections **4a / 4b / 4c** are independent — you can run them in any order or
re-run individually. All three are resume-aware: already-cached files are skipped.

---

## 1. Install dependencies

In [1]:
%%capture
!pip install mne moabb scikit-learn pyyaml scipy

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data cached here — avoids re-downloading across sessions
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.makedirs(MNE_DATA_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Checkpoints saved here
CKPT_ROOT = f'{DRIVE_ROOT}/runs'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Drive mounted. Checkpoints → {CKPT_ROOT}')
# Preprocessing cache (post-bandpass, resample, epoch, normalize).
# Set so dataset loaders cache to Drive — built once here, reused
# by every pretrain + eval run.
CACHE_ROOT = f'{DRIVE_ROOT}/cache'
os.makedirs(CACHE_ROOT, exist_ok=True)
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
print(f'Cache dir → {CACHE_ROOT}')


Mounted at /content/drive
Drive mounted. Checkpoints → /content/drive/MyDrive/geometric_eeg_ssl/runs
Cache dir → /content/drive/MyDrive/geometric_eeg_ssl/cache


## 3. Clone repo (optional, for the loader code)

In [3]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR)

Cloning into '/content/geometric-eeg-ssl'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 202 (delta 94), reused 166 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 176.00 KiB | 1.31 MiB/s, done.
Resolving deltas: 100% (94/94), done.
Repo ready at /content/geometric-eeg-ssl


## 4a. Download PhysioNet MI data

In [4]:
import os, mne
from concurrent.futures import ThreadPoolExecutor, as_completed

mne.set_log_level('WARNING')

EXCLUDED = {88, 92, 100, 104}
ALL_SUBJECTS_MI = [s for s in range(1, 110) if s not in EXCLUDED]  # 105 subjects
MI_RUNS = [4, 6, 8, 10, 12, 14]

EEGBCI_ROOT = os.path.join(MNE_DATA_DIR, 'MNE-eegbci-data', 'files', 'eegmmidb', '1.0.0')

def subject_fully_cached(subj, runs):
    subj_dir = os.path.join(EEGBCI_ROOT, f'S{subj:03d}')
    if not os.path.isdir(subj_dir):
        return False
    return all(
        os.path.exists(os.path.join(subj_dir, f'S{subj:03d}R{run:02d}.edf')) and
        os.path.getsize(os.path.join(subj_dir, f'S{subj:03d}R{run:02d}.edf')) > 0
        for run in runs
    )

cached = [s for s in ALL_SUBJECTS_MI if subject_fully_cached(s, MI_RUNS)]
todo = [s for s in ALL_SUBJECTS_MI if s not in cached]
print(f'PhysioNet MI: cached {len(cached)}/{len(ALL_SUBJECTS_MI)} subjects. '
      f'Downloading {len(todo)} in parallel...')

N_WORKERS = 8  # I/O-bound; pooch atomic-moves make concurrent writes safe

def _fetch_one(subj):
    try:
        mne.datasets.eegbci.load_data(subj, MI_RUNS, path=MNE_DATA_DIR, verbose=False)
        return subj, None
    except Exception as e:
        return subj, str(e)

completed = 0
n_failed = 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(_fetch_one, s): s for s in todo}
    for fut in as_completed(futures):
        completed += 1
        subj, err = fut.result()
        if err is not None:
            n_failed += 1
            print(f'  subject {subj:3d}: failed ({err[:80]})')
        if completed % 10 == 0 or completed == len(todo):
            print(f'  [{completed:3d}/{len(todo)}] last completed: subject {subj}')

still_missing = [s for s in ALL_SUBJECTS_MI if not subject_fully_cached(s, MI_RUNS)]
if still_missing:
    print(f'WARNING: {len(still_missing)} subjects still incomplete: {still_missing}')
    print('Re-run this cell to resume; partial files are skipped automatically.')
else:
    print(f'All {len(ALL_SUBJECTS_MI)} PhysioNet MI subjects ready.')

PhysioNet MI: cached 105/105 subjects. Downloading 0 in parallel...
All 105 PhysioNet MI subjects ready.


## 4b. Download BCIC-2B data

In [5]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

from moabb.datasets import BNCI2014_004
import moabb
moabb.set_log_level('WARNING')

ds = BNCI2014_004()
print('Downloading BCIC-2B (BNCI2014_004)...')
try:
    ds.download(subject_list=list(range(1, 10)))
    print('BCIC-2B download complete.')
except Exception as e:
    # MOABB sometimes raises on partial cache; data may still be usable
    print(f'MOABB download reported: {e}')
    print('Attempting to load subject 1 to verify cache...')
    try:
        _ = ds.get_data(subjects=[1])
        print('Subject 1 loaded OK — cache is usable.')
    except Exception as e2:
        print(f'WARNING: could not load subject 1: {e2}')

/usr/local/lib/python3.12/dist-packages/moabb/datasets/download.py:97: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))


BCIC-2B download complete.


## 4c. Download Sleep-EDFx data

In [6]:
import os, mne
from concurrent.futures import ThreadPoolExecutor, as_completed

mne.set_log_level('WARNING')
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Resume-aware: pooch hash-checks every file before downloading and skips
# files that are already on Drive — re-running this cell only re-attempts
# the missing/failed ones. Each night is fetched independently so a
# night-2 throttle doesn't cause us to re-download night 1.
_UNAVAILABLE = {39, 68, 69, 78, 79}
ALL_SUBJECTS_SLEEP = [s for s in range(0, 83) if s not in _UNAVAILABLE]

N_WORKERS = 4  # PhysioNet rate-limits aggressive parallelism; 4 is the sweet spot
print(f'Sleep-EDFx: fetching {len(ALL_SUBJECTS_SLEEP)} subjects with '
      f'{N_WORKERS} parallel workers (cached subjects skip instantly)...')

def _fetch_one(subj):
    """Fetch night 1 and night 2 independently. Returns (subj, status)."""
    n1_ok = n2_ok = False
    try:
        mne.datasets.sleep_physionet.age.fetch_data(
            subjects=[subj], recording=[1], path=MNE_DATA_DIR, verbose=False
        )
        n1_ok = True
    except Exception:
        pass
    try:
        mne.datasets.sleep_physionet.age.fetch_data(
            subjects=[subj], recording=[2], path=MNE_DATA_DIR, verbose=False
        )
        n2_ok = True
    except Exception:
        pass
    if n1_ok and n2_ok:
        return subj, 'both'
    if n1_ok:
        return subj, 'night1_only'
    if n2_ok:
        return subj, 'night2_only'
    return subj, 'failed'

n_done = n_failed = 0
completed = 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(_fetch_one, s): s for s in ALL_SUBJECTS_SLEEP}
    for fut in as_completed(futures):
        completed += 1
        subj, status = fut.result()
        if status == 'failed':
            n_failed += 1
        else:
            n_done += 1
        if completed % 5 == 0 or completed == len(ALL_SUBJECTS_SLEEP):
            print(f'  [{completed:3d}/{len(ALL_SUBJECTS_SLEEP)}] '
                  f'ok={n_done}  failed={n_failed}  (last: subject {subj}, {status})')

print(f'\nSleep-EDFx: {n_done} subjects cached, {n_failed} failed. '
      f'Re-run this cell to retry failures (already-cached files will skip).')

Sleep-EDFx: fetching 78 subjects with 4 parallel workers (cached subjects skip instantly)...


  [  5/78] ok=5  failed=0  (last: subject 4, both)
  [ 10/78] ok=10  failed=0  (last: subject 10, both)


  [ 15/78] ok=15  failed=0  (last: subject 8, both)


  [ 20/78] ok=20  failed=0  (last: subject 19, both)
  [ 25/78] ok=25  failed=0  (last: subject 27, both)
  [ 30/78] ok=30  failed=0  (last: subject 32, both)
  [ 35/78] ok=35  failed=0  (last: subject 37, both)
  [ 40/78] ok=40  failed=0  (last: subject 43, both)
  [ 45/78] ok=45  failed=0  (last: subject 48, both)
  [ 50/78] ok=50  failed=0  (last: subject 53, both)
  [ 55/78] ok=55  failed=0  (last: subject 58, both)
  [ 60/78] ok=60  failed=0  (last: subject 63, both)
  [ 65/78] ok=65  failed=0  (last: subject 70, both)
  [ 70/78] ok=70  failed=0  (last: subject 75, both)
  [ 75/78] ok=75  failed=0  (last: subject 82, both)


  [ 78/78] ok=78  failed=0  (last: subject 21, both)

Sleep-EDFx: 78 subjects cached, 0 failed. Re-run this cell to retry failures (already-cached files will skip).


## 5. Build preprocessing caches (CPU)

After raw data is downloaded, run each loader once to produce a cached
`.npz` of preprocessed arrays under `{DRIVE_ROOT}/cache/`. Each subsequent
pretrain or eval run loads from this cache in ~seconds instead of
re-doing ~15 minutes of bandpass + resample + epoch + normalize per dataset.

Cache key is a hash of the preprocessing config — change any preprocessing
knob (sample rate, bandpass, epoch length, etc.) and a fresh cache will
be built automatically; the old file remains until you delete it.

Each subsection below is independent and skippable. Re-running a cell with
an already-built cache prints `[cache] loading ...` and exits in seconds.

### 5a. PhysioNet MI cache (pretrain mode — 105 subjects)

In [7]:
import os, sys
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')

from config import Config
from datasets.physionet_mi import PhysioNetMI, EXCLUDED_SUBJECTS

cfg = Config()
ALL_MI = [s for s in range(1, 110) if s not in EXCLUDED_SUBJECTS]
print(f'Building PhysioNet MI cache for {len(ALL_MI)} subjects (pretrain mode)...')
X, y, ch_pos, ch_names = PhysioNetMI(subjects=ALL_MI, cfg=cfg,
                                     mode='pretrain', verbose=True).load()
print(f'\nX: {X.shape}  y: {y.shape}  ch_pos: {ch_pos.shape}  ch_names: {len(ch_names)}')

Building PhysioNet MI cache for 105 subjects (pretrain mode)...
[cache] building physionet_mi/pretrain; will save to physionet_mi_pretrain_1bfb941d013e1e1e.npz
Do you want to set the path:
    /content/drive/MyDrive/geometric_eeg_ssl/mne_data
as the default EEGBCI dataset path in the mne-python config [y]/n? y
  subject   1 run  4: 15 epochs
  subject   1 run  8: 15 epochs
  subject   1 run 12: 15 epochs
  subject   1 run  6: 15 epochs
  subject   1 run 10: 15 epochs
  subject   1 run 14: 15 epochs
  subject   2 run  4: 15 epochs
  subject   2 run  8: 15 epochs
  subject   2 run 12: 15 epochs
  subject   2 run  6: 15 epochs
  subject   2 run 10: 15 epochs
  subject   2 run 14: 15 epochs
  subject   3 run  4: 15 epochs
  subject   3 run  8: 15 epochs
  subject   3 run 12: 15 epochs
  subject   3 run  6: 15 epochs
  subject   3 run 10: 15 epochs
  subject   3 run 14: 15 epochs
  subject   4 run  4: 15 epochs
  subject   4 run  8: 15 epochs
  subject   4 run 12: 15 epochs
  subject   4 ru

### 5b. PhysioNet MI cache (eval mode — 105 subjects)

Eval mode applies amplitude rejection (currently disabled by default; same
arrays as pretrain mode) and disables clip-by-sigma. Cached separately
because the fingerprint differs.

In [8]:
import os, sys
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')

from config import Config
from datasets.physionet_mi import PhysioNetMI, EXCLUDED_SUBJECTS

cfg = Config()
ALL_MI = [s for s in range(1, 110) if s not in EXCLUDED_SUBJECTS]
print(f'Building PhysioNet MI cache for {len(ALL_MI)} subjects (eval mode)...')
X, y, ch_pos, ch_names = PhysioNetMI(subjects=ALL_MI, cfg=cfg,
                                     mode='eval', verbose=False).load()
print(f'X: {X.shape}  y: {y.shape}')

Building PhysioNet MI cache for 105 subjects (eval mode)...
[cache] building physionet_mi/eval; will save to physionet_mi_eval_a0044464e0ee3ec7.npz
[cache] saved physionet_mi_eval_a0044464e0ee3ec7.npz (1786.8 MB)
X: (9408, 64, 800)  y: (9408,)


### 5c. BCIC-2B cache (eval mode — 9 subjects)

In [9]:
import os, sys
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')

from config import Config
from datasets.bcic_2b import BCIC2B, ALL_SUBJECTS

cfg = Config()
print(f'Building BCIC-2B cache for {len(ALL_SUBJECTS)} subjects (eval mode)...')
X, y, ch_pos, ch_names = BCIC2B(subjects=list(ALL_SUBJECTS), cfg=cfg,
                                mode='eval', verbose=False).load()
print(f'X: {X.shape}  y: {y.shape}  ch_pos: {ch_pos.shape}')

Building BCIC-2B cache for 9 subjects (eval mode)...
[cache] building bcic_2b/eval; will save to bcic_2b_eval_bab5e6065c3c41f0.npz
[cache] saved bcic_2b_eval_bab5e6065c3c41f0.npz (58.2 MB)
X: (6520, 3, 800)  y: (6520,)  ch_pos: (3, 3)


### 5d. Sleep-EDFx cache (eval mode — all available subjects)

This is the longest cache build: ~78 subjects × 2 nights of 30 s
hypnogram-annotated EEG, sliced into 4 s sub-epochs. Expect ~20–40 min.

In [10]:
import os, sys
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')

from config import Config
from datasets.sleep_edfx import SleepEDFx, ALL_SUBJECTS

cfg = Config()
print(f'Building Sleep-EDFx cache for {len(ALL_SUBJECTS)} subjects (eval mode)...')
X, y, ch_pos, ch_names, night = SleepEDFx(subjects=list(ALL_SUBJECTS), cfg=cfg,
                                          mode='eval', verbose=True).load()
print(f'\nX: {X.shape}  y: {y.shape}  night: {night.shape}')

Building Sleep-EDFx cache for 78 subjects (eval mode)...
[cache] building sleep_edfx/eval; will save to sleep_edfx_eval_30e7a0bdafc589da.npz


ValueError: Requested recording 1 for subject 36 and/or 52, but it is not available in corpus.

## Done

Close this runtime to free CPU resources. Both raw downloads AND preprocessing
caches now live on Drive. Open `colab_pretrain.ipynb` with a **GPU runtime**
to start training — it will read `{DRIVE_ROOT}/cache/` directly and skip the
~15 min preprocessing per variant.